# Multi-Agent RAG System with LangGraph
## End-to-End Demo

This notebook walks through the complete pipeline:
1. Environment setup
2. Document ingestion (PDF → FAISS + BM25)
3. Running a single query through the agent graph
4. Inspecting the LangGraph state at each step
5. Viewing the LangSmith trace

In [ ]:
# 0. Install dependencies (run once)
# !pip install -e ".[dev]"

In [ ]:
import sys
sys.path.insert(0, '../src')  # allow imports from src/ without pip install

from dotenv import load_dotenv
load_dotenv('../.env')

from multi_agent_rag.config import settings
print('OpenAI key set:', bool(settings.openai_api_key))
print('LangSmith tracing:', settings.langchain_tracing_v2)
print('LangSmith project:', settings.langchain_project)

## Step 1 — Ingest Financial PDFs

Place PDF files in `../data/sample_reports/` before running this cell.

In [ ]:
from multi_agent_rag.vectorstore.ingestion import build_vectorstore

docs = build_vectorstore(
    pdf_dir='../data/sample_reports',
    index_path='../faiss_index',
)
print(f'Ingested {len(docs)} chunks')
print('Sample chunk:', docs[0].page_content[:200])

## Step 2 — Build Hybrid Retriever

Combines FAISS (semantic) + BM25 (keyword) using Reciprocal Rank Fusion.

In [ ]:
from multi_agent_rag.vectorstore.retriever import build_ensemble_retriever
from multi_agent_rag.tools.search_tool import _set_retriever

retriever = build_ensemble_retriever(index_path='../faiss_index')
_set_retriever(retriever)

# Quick retrieval test
results = retriever.invoke('quarterly revenue earnings per share')
print(f'Retrieved {len(results)} chunks')
for r in results[:2]:
    print('\n---')
    print(r.page_content[:300])

## Step 3 — Run the Full Agent Graph

The graph runs: **supervisor → retrieval → reasoning → validation → response**

In [ ]:
from multi_agent_rag.graph.builder import build_graph
from langchain_core.messages import HumanMessage

graph = build_graph(use_checkpointer=True)

QUERY = 'What was the total revenue and net income for the most recent quarter?'

initial_state = {
    'query': QUERY,
    'chat_history': [HumanMessage(content=QUERY)],
    'retrieved_docs': [],
    'sources': [],
    'reasoning_output': '',
    'validated_answer': '',
    'validation_score': 0.0,
    'next_agent': 'retrieval',
    'retry_count': 0,
    'error': None,
}

config = {'configurable': {'thread_id': 'demo-001'}}
result = graph.invoke(initial_state, config=config)

print('=' * 60)
print('ANSWER')
print('=' * 60)
print(result['validated_answer'])

## Step 4 — Inspect Graph State

In [ ]:
print(f"Validation score : {result['validation_score']:.0%}")
print(f"Retry count      : {result['retry_count']}")
print(f"Sources          : {result['sources']}")
print(f"Retrieved chunks : {len(result['retrieved_docs'])}")
print(f"Chat history len : {len(result['chat_history'])}")

## Step 5 — Multi-turn Follow-up Question

The `InMemorySaver` checkpointer preserves state across invocations on the same `thread_id`.

In [ ]:
FOLLOWUP = 'How does that compare to the same quarter last year?'

followup_state = {
    'query': FOLLOWUP,
    'chat_history': [HumanMessage(content=FOLLOWUP)],
    'retrieved_docs': [],
    'sources': [],
    'reasoning_output': '',
    'validated_answer': '',
    'validation_score': 0.0,
    'next_agent': 'retrieval',
    'retry_count': 0,
    'error': None,
}

result2 = graph.invoke(followup_state, config=config)  # same thread_id
print(result2['validated_answer'])

## Step 6 — View LangSmith Trace

If `LANGCHAIN_TRACING_V2=true` is set, open [smith.langchain.com](https://smith.langchain.com)
and navigate to your project `multi-agent-rag` to see the full trace.

The trace shows every node with:
- Input / output at each step
- Token usage and latency
- LLM calls inside reasoning and supervisor nodes